# 05 · Cruce general y configurable — cualquier par de bases

**Que hace**: cruza (o deduplica) **N fuentes locales** con nombres de columna,
separadores, encoding y tipos **distintos**, y deja en el `workspace`
`golden.parquet` (una fila por entidad), `correlativa.parquet` (cada fila original
con su `ID_GRUPO`), `golden.xlsx`/`correlativa.xlsx` si caben, y
`metadatos_corrida.json` con los parametros exactos de la corrida.

**Como se usa**: ejecute la Celda A una vez, revise la Celda A1 (rutas), luego la
Celda A2, y edite **solo la Celda B**. La Celda C es revision opcional.

**Diseñado para Google Colab Free**: los insumos que esten en Drive se copian
una sola vez al disco local de la VM, el trabajo (checkpoints SQLite) nunca
ocurre sobre Drive, y los resultados se escriben en el `workspace` que usted
indique. Si la sesion se cae, vuelva a ejecutar: los checkpoints se reutilizan.

> Version de la libreria: **0.22.4**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A — ENTORNO  (ejecutar UNA vez por sesion; no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════
# La libreria ya esta descomprimida en Drive. Se COPIA a disco local y se
# instala desde ahi: construir sobre Drive es lento, deja artefactos en su
# carpeta de datos y a veces falla por FUSE.
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

VERSION_OBJETIVO = "0.22.4"

# Carpeta donde descomprimio la libreria (la que contiene pyproject.toml).
RUTA_LIBRERIA = Path("/content/drive/MyDrive/rues_linker_pruebas")
COPIAR_A_LOCAL = True  # False solo si la libreria no esta en Drive

# ── 1. Drive ──────────────────────────────────────────────────────────────
if Path("/content").is_dir() and not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive

    drive.mount("/content/drive")

# ── 2. La libreria esta donde decimos ─────────────────────────────────────
_REQUERIDOS = (
    RUTA_LIBRERIA / "pyproject.toml",
    RUTA_LIBRERIA / "src" / "record_linkage" / "__init__.py",
)
_faltantes = [str(r) for r in _REQUERIDOS if not r.is_file()]
if _faltantes:
    raise FileNotFoundError(
        "No encuentro la libreria en RUTA_LIBRERIA. Falta:"
        + "".join("\n- " + f for f in _faltantes)
        + "\n\nDescomprima el zip y apunte RUTA_LIBRERIA a la carpeta que "
        "contiene pyproject.toml (en el zip se llama rues_linker_pruebas)."
    )

# ── 3. Copia a disco local ────────────────────────────────────────────────
_origen = RUTA_LIBRERIA
if COPIAR_A_LOCAL and Path("/content").is_dir():
    _origen = Path("/content/rues_linker_lib")
    if _origen.exists():
        shutil.rmtree(_origen)
    shutil.copytree(
        RUTA_LIBRERIA,
        _origen,
        ignore=shutil.ignore_patterns(
            ".git",
            "__pycache__",
            "*.egg-info",
            "build",
            "dist",
            ".pytest_cache",
            ".ruff_cache",
            ".mypy_cache",
        ),
    )
    print(f"Libreria copiada a {_origen}")

# ── 4. Instalacion ────────────────────────────────────────────────────────
# NO se usa `pip install -e`. Un editable moderno deja un archivo .pth que
# solo se lee AL ARRANCAR el interprete: en un kernel ya en marcha pip
# devuelve 0 y aun asi `import record_linkage` falla con ModuleNotFoundError.
# Una instalacion normal copia el paquete a site-packages y se ve de inmediato.
_objetivo = str(_origen)
print(f"Instalando desde el fuente auditado: {Path(_objetivo).name} ...")
_proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", _objetivo],
    capture_output=True,
    text=True,
)
if _proc.returncode != 0:
    print(_proc.stdout[-3000:])
    print(_proc.stderr[-3000:])
    raise RuntimeError(
        "pip install fallo; el detalle esta arriba. Casi siempre es una "
        "dependencia que no pudo bajarse."
    )

# ── 5. Que el kernel YA en marcha vea únicamente esta versión ─────────────
sys.path.insert(0, str(_origen / "src"))
for _modulo in [
    m for m in list(sys.modules) if m == "record_linkage" or m.startswith("record_linkage.")
]:
    del sys.modules[_modulo]
importlib.invalidate_caches()
try:
    import record_linkage
except ModuleNotFoundError:
    sys.path.insert(0, str(_origen / "src"))  # plan B: apuntar al fuente
    importlib.invalidate_caches()
    try:
        import record_linkage
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "La libreria se instalo pero este kernel no la ve. Reinicie el "
            "entorno de ejecucion (Entorno de ejecucion > Reiniciar sesion) y "
            "vuelva a ejecutar esta celda."
        ) from exc

if record_linkage.__version__ != VERSION_OBJETIVO:
    raise RuntimeError(
        f"Se cargó rues-linker {record_linkage.__version__}, pero se requiere {VERSION_OBJETIVO}. "
        "Reinicie el entorno y vuelva a ejecutar esta celda."
    )

# ── 6. Que el paquete esté COMPLETO, no solo importable ──────────────────
# Importar el __init__ no prueba nada sobre los submódulos. El árbol se copia
# desde Drive por FUSE, que sincroniza de forma asíncrona: la copia puede
# quedar incompleta sin lanzar error y pip instala lo que haya. El síntoma
# —ModuleNotFoundError de un submódulo DESPUÉS de que pip dijo que instaló—
# no apunta a la causa. Dos segundos aquí ahorran ese diagnóstico.
_SUBPAQUETES = (
    "config.paths",
    "flujo.cruce",
    "processing.text",
    "matching.campos",
    "engine.scorer",
    "pipeline.result",
)
_rotos = []
for _sub in _SUBPAQUETES:
    try:
        importlib.import_module(f"record_linkage.{_sub}")
    except Exception as _exc:  # noqa: BLE001 - el mensaje ES el producto
        _rotos.append(f"{_sub}: {type(_exc).__name__}: {_exc}")
if _rotos:
    raise ImportError(
        "Qué pasó: el paquete se instaló pero le faltan submódulos:\n"
        + "".join(f"  - {r}\n" for r in _rotos)
        + "Por qué importa: la corrida fallaría a mitad de camino.\n"
        "Qué hacer: espere a que Drive termine de sincronizar y repita esta "
        "celda; si persiste, reinicie la sesión y vuelva a descomprimir el .zip."
    )

print(f"rues-linker {record_linkage.__version__} listo | Python {sys.version.split()[0]}")
print(f"Cargado desde: {Path(record_linkage.__file__).parent}")
print(f"Submódulos verificados: {len(_SUBPAQUETES)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A1 — RUTAS DE DATOS  (donde estan sus archivos y donde quedan las salidas)
# ══════════════════════════════════════════════════════════════════════════
# Por defecto los insumos estan en Drive: la libreria los copia una sola vez
# al disco local de la VM y el trabajo (checkpoints SQLite) nunca ocurre
# sobre Drive. Los nombres de archivo y el mapeo de columnas van en la Celda B.
# tests/test_notebooks_contrato.py sobreescribe estas rutas para correr el
# notebook sobre datos sinteticos.
from pathlib import Path

RUTA_DATOS = Path("/content/drive/MyDrive/mis_datos")  # carpeta con base_a.zip y base_b.txt
RUTA_RESULTADOS = RUTA_DATOS / "resultados"  # aqui quedan las salidas (workspace)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA A2 — EXTRAS  (definiciones; no requiere edicion)
# ══════════════════════════════════════════════════════════════════════════
# Toda la logica del flujo (preflight, copia anti-Drive, smoke test,
# invariantes, exportes, metadatos) vive en `record_linkage.flujo`, que se
# entrega con pruebas. Aqui solo se importan los nombres que usara la Celda B.
import gc

import pandas as pd

from record_linkage import (
    ColumnType,
    Compression,
    IdentifierFormat,
    SourceSpec,
    load_source,
)
from record_linkage.flujo import (
    ConfigCruce,
    diagnosticar_identificadores,
    ejecutar_cruce,
)


def explorar(spec: SourceSpec, filas: int = 5) -> pd.DataFrame:
    """Carga la fuente y muestra las primeras filas ya mapeadas al canonico.

    Uselo ANTES de la corrida completa: si el mapeo de columnas esta mal, se
    ve aqui en segundos en vez de descubrirlo tras una hora de proceso.
    """
    cargada = load_source(spec)
    print(f"{spec.name}: {cargada.report.rows:,} filas x {len(cargada.report.columns)} columnas")
    print(
        f"  motor={cargada.report.engine} encoding={cargada.report.encoding} "
        f"sep={cargada.report.delimiter!r}"
    )
    if cargada.report.invalid_values:
        print(f"  valores invalidos por columna: {cargada.report.invalid_values}")
    return cargada.data.head(filas)


def liberar(*variables) -> None:
    """Libera memoria explicitamente entre fases largas."""
    for variable in variables:
        del variable
    gc.collect()
    try:
        import psutil

        ram = psutil.virtual_memory()
        print(f"RAM libre: {ram.available / 1024**3:.1f} GB de {ram.total / 1024**3:.1f} GB")
    except ImportError:
        print("gc.collect() ejecutado")


print("Extras cargados. Edite unicamente la Celda B.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA B — EJECUTAR  (la unica celda que usted edita)
# ══════════════════════════════════════════════════════════════════════════
# 1) Declare UNA SourceSpec por archivo. Los nombres, separadores, encoding y
#    tipos pueden ser distintos entre fuentes: para eso existe el contrato.
FUENTES = [
    SourceSpec(
        name="FUENTE_A",
        path=RUTA_DATOS / "base_a.zip",
        compression=Compression.ZIP,  # AUTO tambien funciona
        delimiter=",",
        encoding="auto",  # detecta UTF-8 / CP1252
        column_mapping={  # {canonico: nombre_en_el_archivo}
            "NIT": "NUMERO_IDENTIFICACION",
            "RAZON_SOCIAL": "RAZON_SOCIAL",
        },
        optional_column_mapping={"CIUDAD": "CODIGO_MUNICIPIO_COMERCIAL"},
        column_types={"NIT": ColumnType.IDENTIFIER, "RAZON_SOCIAL": ColumnType.STRING},
        identifier_formats={"NIT": IdentifierFormat(mode="digits", min_length=6, max_length=12)},
    ),
    SourceSpec(
        name="FUENTE_B",
        path=RUTA_DATOS / "base_b.txt",
        delimiter="\t",
        column_mapping={"NIT": "Nit", "RAZON_SOCIAL": "Razon Social"},
        column_types={"NIT": ColumnType.IDENTIFIER},
        null_values={"NIT": ("-1", "0"), "RAZON_SOCIAL": ("NO DEFINIDO", "SIN DATO")},
    ),
]

# 2) Parametros de la corrida.
CFG = ConfigCruce(
    fuentes=FUENTES,
    workspace=RUTA_RESULTADOS,  # aqui quedan las salidas (Celda A1)
    confiables={"FUENTE_A"},  # fuentes con identificador verificado
    col_ciudad=None,  # "CIUDAD" si la mapeo en ambas fuentes
    variables_extra=None,  # p. ej. ["TELEFONO", "EMAIL"]
    filas_smoke=4_000,  # ensayo previo; 0 lo desactiva
    exportar_excel=True,
)

# 3) Ejecutar.
resultado = ejecutar_cruce(CFG)
print(resultado.resumen())
print(resultado.cruce_por_fuente().to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CELDA C — REVISION (opcional)
# ══════════════════════════════════════════════════════════════════════════
# Antes de una corrida grande conviene ver que trae cada fuente y cuantos
# identificadores llegan mal: es la diferencia entre un cruce bajo explicable
# y un cruce bajo inexplicable.
display(explorar(FUENTES[0]))
print(diagnosticar_identificadores(resultado.correlativa["NIT"]).resumen())

# Entidades presentes en mas de una fuente (el resultado de negocio).
corr = resultado.correlativa
fuentes_por_grupo = corr.groupby("ID_GRUPO")["SRC"].transform("nunique")
display(corr[fuentes_por_grupo > 1].sort_values("ID_GRUPO").head(20))

## Notas operativas

- **Solo mapee las columnas que necesita.** Proyectar 2 de 42 columnas es la
  diferencia entre 3 MB y 300 MB de RAM por fuente.
- **`confiables`**: fuentes cuyo identificador esta verificado. No se
  deduplican internamente y mandan en el nombre canonico del golden.
- **`null_values`**: declare aqui los centinelas reales de sus archivos
  (`-1`, `NO DEFINIDO`, `SIN DATO`). Si no, entran como si fueran datos.
- **Que queda en el `workspace`**: `golden.parquet`, `correlativa.parquet`,
  `metadatos_corrida.json` y, con `exportar_excel=True`, `golden.xlsx` y
  `correlativa.xlsx` — por encima de 500.000 filas el Excel no se escribe
  (limite del formato); el Parquet siempre queda. La carpeta del estandar de
  salida con `manifest.json`, `diccionario.csv` y `excel/` la escriben hoy
  `linkage(carpeta_salida=...)`, `link()` y `dedupe()` (notebooks 01–03); este
  flujo la adoptara en una version posterior.
- **Reanudar**: mantenga el mismo `workspace`. Para forzar todo de nuevo,
  `ConfigCruce(..., reusar_checkpoints=False)`.
- **Deduplicar una sola base**: declare una unica fuente en `FUENTES`.